# 4c - The long-only portfolio, with AI

**Time:** 45 minutes build + 20 minutes check  |  **AI: builds a solver from YOUR spec** - you verify it

**What you will do**
- Write a spec where the **Constraints** line decides the answer.
- Ask Gemini to write a solver from your spec. It uses `minimize` from SciPy, which finds the inputs that make a function as small as possible. Paste it here, three times, with three versions of the spec.
- See how the answer changes with the constraints, and compare with the brute force from Session 2.
- Run the same solver on five real stocks.
- Then check the answer: the status, the weights, a second method, and your own eyes.

The code is written by AI. The decision is yours: what you ask for decides what you get.

## The problem

In Session 2 you found the lowest-risk portfolio of funds A, B and C two ways, and got two answers: a clever formula (72 / 50 / -22) and a brute force over every long-only portfolio, where no weight is below 0 (65 / 35 / 0). The spec decided the answer.

Today a **solver** does the job. A solver does not try every portfolio. It starts from a guess and walks downhill to the lowest risk.

Why a solver? The brute force tries every portfolio in steps of 1%:

| Assets | Portfolios to try |
|---|---|
| 3 funds (Session 2) | 5,151 |
| 5 stocks | 4.6 million |
| 20 stocks | about 5 x 10^21 |

Same five lines as in Session 2. Today the **Constraints** line (units, conventions and the rules the answer must obey) decides everything. This is the spec for `port_vol`, the function you are given below:

| | The example |
|---|---|
| **Goal** | Volatility of a portfolio |
| **Inputs** | `weights` (array), `cov` (annual covariance matrix) |
| **Constraints** | decimals (0.15 = 15%), annual; no rules: this is a calculation, not a search |
| **Output** | one number, the annual volatility |
| **Check** | one fund at 100% gives the vol of that fund |

## Given - the three funds and the functions

Same funds as in 2a, 2b and 2c: vols 15%, 20%, 25%, with the same correlations.

Read the code below. The solver needs two functions (`port_vol` and `fully_invested`) and one starting guess (`w0`). It takes the functions **as inputs**, with no round brackets after the name, as `brentq` did in 4a. `show` only prints.
- `port_vol(weights)`: the volatility of a portfolio. It is the function the solver tries to make small.
- `fully_invested(weights)`: the sum of the weights minus 1. It is zero when the weights add up to 100%.
- `w0`: the starting guess, one third in each fund.
- `show(result)`: prints the answer of the solver (we use it below). `minimize` gives back a result: `result.x` holds the best weights, `result.fun` the lowest value found (here the volatility), and `result.success` is True or False: did the solver finish properly?

**Why does `port_vol` have no guard?** In 2b, `portfolio_vol` stopped with `raise ValueError` when the weights did not sum to 1. Here that would break the solver: on its way downhill it tries weights that do not sum to 1. The solver gets a **constraint** instead. The constraint replaces the guard.

In [ ]:
import numpy as np

# Given: the same funds as in 2a, 2b and 2c
vol = np.array([0.15, 0.20, 0.25])    # volatilities of A, B, C
corr = np.array([[1.0, 0.1, 0.4],
                 [0.1, 1.0, 0.7],
                 [0.4, 0.7, 1.0]])
cov = np.diag(vol) @ corr @ np.diag(vol)    # covariance matrix, as in 2a


# Given: the function the solver makes small. No guard: see the note above.
def port_vol(weights):
    return np.sqrt(weights @ cov @ weights)


# Given: zero when the weights sum to 1
def fully_invested(weights):
    return np.sum(weights) - 1


# Given: the starting guess, one third in each fund
w0 = np.array([1 / 3, 1 / 3, 1 / 3])


# Given: print what the solver found (zip pairs each name with its weight; the names are A, B, C unless you give your own)
def show(result, names=("A", "B", "C")):
    w = np.round(result.x, 3) + 0.0    # round, and + 0.0 turns -0.0 into 0.0
    for name, weight in zip(names, w):
        print(f"{name:>10}: {weight:7.1%}")
    print(f"Volatility: {port_vol(result.x):.2%}")
    print("success:", result.success)
    print("message:", result.message)

## Step 1 - Three specs, three answers

The three versions of the spec differ in one line only: the **Constraints**.

### Spec 1 - no constraints

Double-click the cell below and fill it in. Be precise: AI fills every gap with its own guess. Use the table in "The problem" as a model: one short line per row. In Inputs, name `port_vol`, `w0` and `cov`. Leave the Constraints line empty on purpose, or write "none".

| | Your spec |
|---|---|
| **Goal** | ... |
| **Inputs** | ... |
| **Constraints** | ... |
| **Output** | ... |
| **Check** | ... |

Ask Gemini (gemini.google.com, or the Gemini button in Colab): paste your spec, then ask *"Write Python code with scipy.optimize.minimize for this spec. Use my function port_vol and my starting guess w0."* Paste the code in the cell below (keep its `import` lines: later cells need `minimize`). Keep the variable name `result_1`.

In [ ]:
# PASTE the code Gemini wrote here.
# port_vol and w0 are defined above. Keep the variable name result_1.
# Then run show(result_1).

**Your answer:** look at the weights, the volatility and the `success` line. What did the solver do? What is missing from the spec? Double-click this cell and write 2 or 3 lines here: ...

### Spec 2 - fully invested

Add one line to the Constraints: the weights sum to 1. In the code, that is the **constraint dictionary**: a Session 1 dictionary with a `"type"` and a `"fun"`. The type `"eq"` means "equal to zero": the solver changes the weights until `fully_invested` gives 0. You hand the dictionary to `minimize` as `constraints=[...]`. The `"fun"` is the function `fully_invested`, again **a function as an input**: no round brackets after its name.

| | Your spec |
|---|---|
| **Goal** | ... |
| **Inputs** | ... |
| **Constraints** | ... |
| **Output** | ... |
| **Check** | ... |

Same method: paste your new spec in Gemini, ask for `scipy.optimize.minimize` code, paste the code below. Keep the variable name `result_2`.

In [ ]:
# PASTE the code Gemini wrote here. Keep the variable name result_2.
# Then run show(result_2).

**Your answer:** compare with Session 2. Which answer is this, the clever formula or the brute force? Is any weight negative? What does a negative weight mean? Write 2 or 3 lines here: ...

### Spec 3 - fully invested and no shorting

Add the line "no weight below 0 or above 100%". This is "no shorting": you cannot sell a fund you do not own. In the code, that is `bounds` (an input of `minimize`): a **list** with one pair `(lowest, highest)` per fund. `[(0, 1)] * 3` is Session 2's **list repetition**: the pair `(0, 1)` written three times.

| | Your spec |
|---|---|
| **Goal** | ... |
| **Inputs** | ... |
| **Constraints** | ... |
| **Output** | ... |
| **Check** | ... |

Paste your spec in Gemini again, and paste the code below. Keep the variable name `result_3`.

In [ ]:
# PASTE the code Gemini wrote here. Keep the variable name result_3.
# Then run show(result_3).

**Your answer:** which weights and which volatility? Which answer from Session 2 does it look like? Write 2 lines here: ...

## Step 2 - Read what the code assumes

You do not need to comment every line. Look at the code in your three paste cells and answer the questions below: they are where a solver can be wrong for your client.

Questions to answer for your own code: What does `minimize` take as its first input? Why no brackets after `port_vol`? What is `w0` for? What does `"eq"` mean? What are `result.x`, `result.fun` and `result.success`?

Double-click here and write: ...

### Brute force again: the Session 2 loop

The same loops as in 2c: try every long-only portfolio in steps of 1%, with `a` in A from 0 to 100, `b` in B from 0 to `100 - a`, and the rest in C. That is 5,151 portfolios. Keep the one with the lowest volatility. This time the function is `port_vol(weights)`, with no guard and no `vol`, `corr` inputs.

In the loops: the rest goes to C, so the three percents add up to 100: save it in `c_pct`. Then the volatility of `trial`, with `port_vol`, in `v`. When it beats the best so far, keep the new volatility in `best_vol` and the weights in `best_w`, and delete the word `pass`.

You should get **65 / 35 / 0**, volatility **12.56%**, 5,151 portfolios.

In [ ]:
best_vol = 1       # start higher than any real vol (100%)
best_w = None
count = 0

for a in range(0, 101):
    for b in range(0, 101 - a):
        # YOUR CODE HERE
        trial = np.array([a, b, c_pct]) / 100    # percent -> decimals
        # YOUR CODE HERE
        count = count + 1
        if v < best_vol:
            # YOUR CODE HERE
            pass

print(f"Portfolios tried: {count:,}")
print(f"Best weights: A {best_w[0]:.0%}, B {best_w[1]:.0%}, C {best_w[2]:.0%}")
print(f"Best volatility: {best_vol:.2%}")

Your solver should say about 65.5 / 34.5 and the brute force says 65 / 35. The gap is the **1% step** of the brute force: it cannot try 65.5. The volatility is the same to two decimals. The solver is not limited to steps of 1%.

## Step 3 - The five mega-caps

Same solver, real data: the five stocks from Session 3 (Apple, Microsoft, Alphabet, Amazon, Meta). The cell below (just run it, no blank) reads the prices, computes the daily returns, then the annual covariance matrix. `returns.cov()` gives the covariance of the daily returns, and `* 252` makes it annual (252 trading days).

From here on, `cov` and `w0` are the five-stock versions. `port_vol` and `fully_invested` do not change: `port_vol` reads `cov`, so it now measures five-stock portfolios. Run the notebook from the top.

In [ ]:
import pandas as pd

DATA = "https://raw.githubusercontent.com/niparis/gem-python-finance/main/data/"
prices = pd.read_excel(DATA + "peers.xlsx", index_col="Date")
stocks = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]    # drop SP500
prices = prices[stocks]
returns = prices.pct_change().dropna()

cov = returns.cov().values * 252      # annual covariance, a 5 x 5 array
w0 = np.ones(5) / 5                   # starting guess: equal weight, 20% each

print(len(returns), "daily returns")
print("Single-stock vols:", dict(zip(stocks, np.sqrt(np.diag(cov)).round(3))))

### Spec 4 - minimum risk, long only

Write the spec for five stocks. The Constraints line is the same as spec 3: sum to 1, each weight between 0 and 1. Think about what changes: `cov` and `w0` are now for five stocks, so how many pairs does `bounds` need?

| | Your spec |
|---|---|
| **Goal** | ... |
| **Inputs** | ... |
| **Constraints** | ... |
| **Output** | ... |
| **Check** | ... |

Paste your spec in Gemini, and paste the code below. Keep the variable name `result_5`. If the code says `[(0, 1)] * 3`, it is wrong for five stocks.

In [ ]:
# PASTE the code Gemini wrote here. Keep the variable name result_5.
# Then run show(result_5, stocks).

You should get Apple **43%**, Microsoft **39%**, Alphabet **18%**, Amazon and Meta **0%**, volatility **24.1%**. Compare with the equal-weight portfolio (20% each): in the cell below, its volatility is **27.1%**. Compute it yourself: the volatility of the starting guess `w0`, with `port_vol`, saved in `equal_vol`.

In [ ]:
# YOUR CODE HERE
print(f"Equal weight volatility: {equal_vol:.2%}")

**Your answer:** which stocks get a weight? Which get zero? Why do you think so? Is the volatility lower than the equal-weight portfolio? Write 3 lines here: ...

## Step 4 (stretch) - The best risk-adjusted mix

Minimum risk ignores returns. The **Sharpe ratio** is the excess return per unit of risk: (return - risk-free rate) / volatility. The best risk-adjusted mix is the portfolio with the highest Sharpe ratio.

You need a return for each stock. Use the **5-year compound annual return**, as in Session 3's risk table: the last price divided by the first price, raised to the power of 252 divided by `days`, minus 1. Here `last` and `first` are the last and first price of each stock, and `days` is the number of daily returns. `iloc` picks a row by its position: the last row, and the first row. Do all five stocks at once, and save the result in `annual` as an array (`.values`). The risk-free rate is 3%: what a safe investment, such as a government bond, pays.

In [ ]:
rf = 0.03                  # risk-free rate, a decimal (3%)
days = len(returns)        # number of daily returns

# YOUR CODE HERE

print(dict(zip(stocks, annual.round(3))))

Check it by another route before you use it: the same return through logarithms: take the log of last divided by first, scale it by 252 over `days`, then undo the log with `np.exp` and take off 1. The two must agree.

In [ ]:
other_way = np.exp(np.log(prices.iloc[-1] / prices.iloc[0]).values * 252 / days) - 1

if not np.allclose(annual, other_way):
    raise ValueError("The annual returns do not agree with the logarithm method.")
print("OK: two methods agree on the annual returns")

### Spec 5 - maximum Sharpe ratio

A solver only **minimizes**. To find the highest Sharpe ratio, minimize the **negative** Sharpe ratio. Your spec must say so, or the AI will not know. Which function does the solver now need as its first input? It is no longer `port_vol`. Name it in your spec, for example `neg_sharpe(weights)`, and say which variables it reads: `annual`, `rf` and `port_vol`.

| | Your spec |
|---|---|
| **Goal** | ... |
| **Inputs** | ... |
| **Constraints** | ... |
| **Output** | ... |
| **Check** | ... |

Paste your spec in Gemini, and paste the code below. Keep the variable name `result_sharpe`. Then run `show(result_sharpe, stocks)`.

In [ ]:
# PASTE the code Gemini wrote here. Keep the variable name result_sharpe.
# annual, rf, port_vol and fully_invested are defined above.
# Then run show(result_sharpe, stocks).

You should get **83% Apple, 17% Alphabet**, nothing else. The cell below computes its Sharpe ratio with your own formula, not the solver's: it should be about **0.60**.

In [ ]:
def sharpe(weights):
    return (weights @ annual - rf) / port_vol(weights)


print(f"Sharpe, max Sharpe mix:  {sharpe(result_sharpe.x):.2f}")
print(f"Sharpe, equal weight:    {sharpe(w0):.2f}")
print(f"Sharpe, minimum risk:    {sharpe(result_5.x):.2f}")

**Is it plausible?** Look at the weights and at the five annual returns above. Why does the solver put so much in one stock? Would you show this portfolio to a client? Write 3 or 4 lines here: ...

### Optional - the frontier

For each target return, the lowest volatility. This is the long-only efficient frontier. It runs `minimize` 30 times, once per target. The `lambda` line builds a tiny function on the spot: you do not need to write one. The cell uses `minimize` from your pasted code. The target return is a second constraint (a second dictionary, in a list).

In [ ]:
import matplotlib.pyplot as plt

full = {"type": "eq", "fun": fully_invested}
targets = np.linspace(result_5.x @ annual, annual.max(), 30)
frontier_vol = []
for target in targets:
    hit_target = {"type": "eq", "fun": lambda w, t=target: w @ annual - t}
    res = minimize(port_vol, w0, bounds=[(0, 1)] * 5, constraints=[full, hit_target])
    frontier_vol.append(res.fun)

plt.figure(figsize=(7, 4.3))
plt.plot(np.array(frontier_vol) * 100, targets * 100, color="black", label="Best mixes, long only")
for i, s in enumerate(stocks):
    plt.plot(np.sqrt(cov[i, i]) * 100, annual[i] * 100, "o")
    plt.text(np.sqrt(cov[i, i]) * 100 + 0.5, annual[i] * 100, s)
plt.plot(port_vol(w0) * 100, (w0 @ annual) * 100, "D", color="grey")
plt.text(port_vol(w0) * 100 + 0.5, (w0 @ annual) * 100, "Equal weight")
plt.plot(port_vol(result_5.x) * 100, (result_5.x @ annual) * 100, "*", color="blue", markersize=14)
plt.plot(port_vol(result_sharpe.x) * 100, (result_sharpe.x @ annual) * 100, "*", color="red", markersize=14)
plt.xlabel("Volatility, % a year")
plt.ylabel("Annual return, % (past 5 years)")
plt.title("Five mega-caps: the best mixes, no shorting (blue: min risk, red: max Sharpe)")
plt.show()

## Check - Is the answer right?

The solver ran, and it printed an answer. That is not a check. Read the **status** first, then the weights, then compare with a second method. (Checks card, entries 7, 8 and 13 to 16.)

Each check below ends with a **guard**: `raise` stops the notebook when the check fails.

### Check 1 - Three funds: the weights and the status

Weights sum to 1, none below 0, `result.success` is True. A solver gives tiny rounding errors, so compare with a small tolerance (`1e-6` is 0.000001), not with exact equality.

Sum the weights of `result_3` (they are `result_3.x`) with `np.sum` and save it in `total`. Find the lowest weight with `np.min` and save it in `lowest`.

In [ ]:
# YOUR CODE HERE
# YOUR CODE HERE

print(f"Sum of the weights: {total:.6f}")
print(f"Lowest weight:      {lowest:.6f}")
print("success:", result_3.success)

if abs(total - 1) > 1e-6:
    raise ValueError(f"The weights sum to {total}, not 1.")
if lowest < -1e-6:
    raise ValueError(f"A weight is negative: {lowest}")
if not result_3.success:
    raise ValueError("The solver did not finish properly: " + result_3.message)
print("OK: fully invested, long only, solver finished")

### What the check catches

The same three checks on the first two answers. The cell runs each check inside `try:`. If a check raises a `ValueError`, Python jumps to `except ValueError` and prints the message, instead of stopping the notebook.

In [ ]:
for name, res in [("Spec 1", result_1), ("Spec 2", result_2)]:
    try:
        if abs(np.sum(res.x) - 1) > 1e-6:
            raise ValueError(f"the weights sum to {np.sum(res.x):.2f}, not 1")
        if np.min(res.x) < -1e-6:
            raise ValueError(f"a weight is negative: {np.min(res.x):.1%}")
        if not res.success:
            raise ValueError("the solver did not finish properly")
        print(name, "passes")
    except ValueError as error:
        print(name, "caught:", error)

### Check 2 - The solver against the brute force

Two methods, one answer. The solver's weights must be close to the brute force's weights. The brute force moves in steps of 1%, so allow a difference of up to 1 percentage point (`0.01`).

Subtract the two arrays of weights (`result_3.x` and `best_w`), take the absolute value of the result with `np.abs`, then its biggest value. Save it in `biggest_gap`.

In [ ]:
# YOUR CODE HERE
print(f"Biggest difference in a weight: {biggest_gap:.1%}")
print(f"Volatility: solver {result_3.fun:.2%}, brute force {best_vol:.2%}")

if biggest_gap > 0.01:
    raise ValueError("The solver and the brute force disagree.")
print("OK: the solver and the brute force agree")

### Check 3 - Five stocks: the weights, the status and the benchmark

There is no brute force for five stocks (4.6 million portfolios is possible but slow, and 20 stocks is impossible). Use the benchmarks you have: the minimum-risk portfolio must have a lower volatility than **every single stock** and than the **equal-weight** portfolio. The max Sharpe portfolio must have a Sharpe ratio at least as high as the equal-weight one.

Run the same weight and status guards on `result_5` and `result_sharpe`. In the code cell, fill the two blanks the same way as in Check 1: the sum of the weights of `res` (`res.x`) in `total5`, and the lowest weight in `lowest5`.

In [ ]:
single_vols = np.sqrt(np.diag(cov))     # the vol of each stock alone

for name, res in [("Min risk", result_5), ("Max Sharpe", result_sharpe)]:
    # YOUR CODE HERE
    # YOUR CODE HERE
    if abs(total5 - 1) > 1e-6:
        raise ValueError(f"{name}: the weights sum to {total5}, not 1.")
    if lowest5 < -1e-6:
        raise ValueError(f"{name}: a weight is negative: {lowest5}")
    if not res.success:
        raise ValueError(f"{name}: the solver did not finish properly: {res.message}")
    print(f"OK: {name} is fully invested, long only, solver finished")

if port_vol(result_5.x) >= min(single_vols):
    raise ValueError("The minimum-risk portfolio is not below every single stock.")
if port_vol(result_5.x) >= equal_vol:
    raise ValueError("The minimum-risk portfolio is not below equal weight.")
print(f"OK: min risk {port_vol(result_5.x):.1%} < lowest single stock {min(single_vols):.1%} and equal weight {equal_vol:.1%}")

if sharpe(result_sharpe.x) < sharpe(w0):
    raise ValueError("The max Sharpe portfolio has a lower Sharpe ratio than equal weight.")
print(f"OK: max Sharpe {sharpe(result_sharpe.x):.2f} >= equal weight {sharpe(w0):.2f}")

### Check 4 - Is it plausible?

Every check above passed, including for the max Sharpe portfolio. Passing is not enough. Look at the largest weight, and at what the solver did to get there.

In [ ]:
for name, weight in zip(stocks, result_sharpe.x):
    print(f"{name:>6}: {weight:6.1%}")
print(f"Largest single weight: {np.max(result_sharpe.x):.0%}")

**Your conclusion.** In your own words: why did the solver and the brute force agree, but the max Sharpe portfolio is still not an answer for a client? What are the four things you check on every solver result? Write 3 or 4 lines here: ...

## What you learned

**Python**
- **Import**: `from scipy.optimize import minimize`.
- **Function as an input**: `minimize(port_vol, w0)`. The solver calls your function many times. No brackets after the name.
- **Constraint dictionary** (Session 1): `{"type": "eq", "fun": fully_invested}`. **Bounds list**: `[(0, 1)] * 3`, a list of pairs, written with Session 2's list repetition.
- **Result object**: `result.x` (the weights), `result.fun` (the value), `result.success` (True or False).
- **For loop** inside a for loop, with `range` and `if v < best_vol`, to try 5,151 portfolios. **Guard** with `raise ValueError`, with a tolerance (`1e-6`) instead of exact equality.
- **Compound annual return**: `(last / first) ** (252 / days) - 1`, as in Session 3.

**Finance**
- A spec has five lines: Goal, Inputs, **Constraints** (units, conventions, rules), Output, Check. The Constraints line decides the answer: none gives "invest nothing" (0 / 0 / 0, `success` False), fully invested gives the short position (72 / 50 / -22, 12.0%), fully invested and long only gives 65 / 35 / 0 (12.56%).
- Read the solver's status, not only its number. A solver gives one answer: ask if there are others.
- The solver agrees with the brute force (65 / 35 / 0), and works for five stocks where the brute force is too slow.
- Five mega-caps, long only: minimum risk is Apple 43%, Microsoft 39%, Alphabet 18%, with a volatility of 24.1% against 27.1% for equal weight.
- The best risk-adjusted mix puts 83% in Apple. It passes every check and is still not plausible: the solver pushes hardest on the weakest input, the past returns.

Keep two pages open from now on:
- [`python-card.md`](https://github.com/niparis/gem-python-finance/blob/main/python-card.md) - what each piece of Python means
- [`checks-card.md`](https://github.com/niparis/gem-python-finance/blob/main/checks-card.md) - how to know a number is right